In [1]:
import pandas as pd 
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, precision_score, recall_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier

In [2]:

df = pd.read_csv('train.csv')
df = df.drop(['id', 'CustomerId', 'Surname' ], axis=1)
df[['NumOfProducts', 'CreditScore', 'Age', 'HasCrCard', 'IsActiveMember', 'Exited', 'Tenure' ]] = df[['NumOfProducts', 'CreditScore', 'Age', 'HasCrCard', 'IsActiveMember', 'Exited', 'Tenure' ]].astype(int)
# df.isna().value_counts() ստուգում ենք NAN արծեքներ կան թե չէ (չկային)

# le =LabelEncoder()
# for_encode = df[["Geography", "Gender"]]

# for col in for_encode:
#     df[col] = le.fit_transform(df[col])

df = pd.get_dummies(df, columns=["Geography"], drop_first=False)

le = LabelEncoder()
df["Gender"] = le.fit_transform(df["Gender"])

x = df.drop(['Exited'], axis=1)
y = df['Exited']
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.25, random_state=42)
for_scale = ['CreditScore', 'Balance', 'EstimatedSalary', 'Age']
scaler = StandardScaler()
x_train[for_scale] = scaler.fit_transform(x_train[for_scale])
x_test[for_scale] = scaler.transform(x_test[for_scale])

In [3]:
y.value_counts()

Exited
0    11948
1     3052
Name: count, dtype: int64

In [3]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, precision_score, recall_score
model_list = {'Lg' : LogisticRegression(random_state=42, class_weight='balanced').fit(x_train, y_train),
              'DC' : DecisionTreeClassifier(random_state=42).fit(x_train, y_train),
              'RF' : RandomForestClassifier(random_state=42, class_weight='balanced').fit(x_train, y_train),
              'XGB': XGBClassifier(random_state=42).fit(x_train, y_train),
              'Cat': CatBoostClassifier(random_state=42, verbose=0).fit(x_train, y_train),
              'LGBM': LGBMClassifier(random_state=42, verbose=-1).fit(x_train, y_train)}

In [4]:
def model_accuracy(model_list):
    for name, model in model_list.items():
        model_pred = model.predict(x_test)
        print(f'{name}: {accuracy_score(y_test, model_pred)}')

model_accuracy(model_list)

Lg: 0.8181333333333334
DC: 0.8501333333333333
RF: 0.8874666666666666
XGB: 0.8856
Cat: 0.8914666666666666
LGBM: 0.8928


In [5]:
def model_f1(model_list):
    for name, model in model_list.items():
        model_pred = model.predict(x_test)
        print(f'{name}: {f1_score(y_test, model_pred)}')
model_f1(model_list)

Lg: 0.6480908152734778
DC: 0.6336375488917861
RF: 0.7365792759051186
XGB: 0.7022900763358778
Cat: 0.7135819845179451
LGBM: 0.7196652719665272


In [ ]:
xgb = XGBClassifier(random_state=42)
param_grid_xgb = {
    'n_estimators': [100, 200, 300],         
     'max_depth': [7, 10],                  
     'learning_rate': [0.01, 0.05, 0.1],      
     'min_child_weight': [1, 3],          
     'subsample': [0.8],          
     'colsample_bytree': [0.8],   
     'scale_pos_weight': [3]   }  

grid = GridSearchCV(xgb, param_grid=param_grid_xgb, cv=5, scoring='f1', n_jobs=-1).fit(x_train, y_train)

best_xgb_params = grid.best_params_
best_xgb_model = grid.best_estimator_

xgb_pred = best_xgb_model.predict(x_test)
print(best_xgb_params)
print(f'f1_xgb : {f1_score(y_test, xgb_pred)}')

{'colsample_bytree': 0.8, 'learning_rate': 0.01, 'max_depth': 10, 'min_child_weight': 3, 'n_estimators': 100, 'scale_pos_weight': 3, 'subsample': 0.8}
f1_xgb : 0.7358247422680413


In [37]:
rf = RandomForestClassifier(random_state=42)
param_grid_rf = {
    'n_estimators': [50, 100, 200],
    'max_depth': [10, 15, 20],
    'min_samples_split': [7,10, 12],
    'min_samples_leaf': [1, 2, 4],
    'class_weight': ['balanced', 'balanced_subsample', None]
} 

grid = GridSearchCV(rf, param_grid=param_grid_rf, cv=5, scoring='f1', n_jobs=-1).fit(x_train, y_train)

best_rf_params = grid.best_params_
best_rf_model = grid.best_estimator_

rf_pred = best_rf_model.predict(x_test)

print(f'f1_rf : {f1_score(y_test, rf_pred)}')
print(best_rf_params)


f1_rf : 0.7383647798742138
{'class_weight': 'balanced_subsample', 'max_depth': 20, 'min_samples_leaf': 1, 'min_samples_split': 10, 'n_estimators': 100}


In [35]:
dc = DecisionTreeClassifier(random_state=42)
param_grid_dc = {
    'max_depth': [ 5, 7, 10, 12, None],
    'min_samples_split': [15, 20, 25],
    'min_samples_leaf': [1, 2, 4, 8],
    'criterion': ['gini', 'entropy'],
    'class_weight': ['balanced']
}

grid = GridSearchCV(dc, param_grid=param_grid_dc, cv=5, scoring='f1', n_jobs=-1).fit(x_train, y_train)

best_dc_params = grid.best_params_
best_dc_model = grid.best_estimator_

dc_pred = best_dc_model.predict(x_test)

print(f'f1_dc : {f1_score(y_test, dc_pred)}')
print(best_dc_params)




f1_dc : 0.7021028037383178
{'class_weight': 'balanced', 'criterion': 'entropy', 'max_depth': 7, 'min_samples_leaf': 2, 'min_samples_split': 25}


In [8]:
lgbm = LGBMClassifier(random_state=42, verbose=-1) 
lgbm_param_grid = {
    'n_estimators': [100, 150],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1],
    'num_leaves': [50, 70],
    'subsample': [0.6, 0.8, 1]
}

grid = GridSearchCV(lgbm, param_grid=lgbm_param_grid, cv=5, scoring='f1', n_jobs=-1, verbose=1).fit(x_train, y_train)

best_lgbm_params = grid.best_params_
best_lgbm_model = grid.best_estimator_

lgbm_pred = best_lgbm_model.predict(x_test)
print(f'f1_lgbm : {f1_score(y_test, lgbm_pred)}')
print(best_lgbm_params)



Fitting 5 folds for each of 72 candidates, totalling 360 fits
f1_lgbm : 0.7196652719665272
{'learning_rate': 0.1, 'max_depth': 5, 'n_estimators': 100, 'num_leaves': 50, 'subsample': 0.6}


In [10]:
cat = CatBoostClassifier(random_state=42, verbose=0)
cat_param_grid = {
    'iterations': [100, 200, 300],
    'depth': [4, 6, 8],
    'learning_rate': [0.01, 0.1, 1],
    'l2_leaf_reg': [1, 3, 5, 7]
}

grid = GridSearchCV(cat, param_grid=cat_param_grid, cv=5, scoring ='f1', n_jobs=-1, verbose=1).fit(x_train, y_train)

best_cat_params = grid.best_params_
best_cat_model = grid.best_estimator_

cat_pred = best_cat_model.predict(x_test)

print(f'f1_cat : {f1_score(y_test, cat_pred)}')
print(best_cat_params)



Fitting 5 folds for each of 108 candidates, totalling 540 fits
f1_cat : 0.7166077738515901
{'depth': 4, 'iterations': 200, 'l2_leaf_reg': 5, 'learning_rate': 0.1}


In [12]:
after_grid = {'xgb_after_grid' : XGBClassifier(colsample_bytree = 0.8, learning_rate = 0.01, max_depth= 10, min_child_weight= 3, n_estimators= 100, scale_pos_weight=3, subsample= 0.8).fit(x_train, y_train),
'lg_after_grid' : LogisticRegression(C= 0.01, penalty = 'l2', solver= 'saga', class_weight='balanced', random_state=42).fit(x_train, y_train),
'dc after_grid' : DecisionTreeClassifier(class_weight='balanced', criterion='entropy', max_depth=7, min_samples_leaf=2,  min_samples_split=25).fit(x_train, y_train),
'lgbm after_grid': LGBMClassifier(random_state=42, verbose=-1, learning_rate= 0.1, max_depth= 5, n_estimators= 100, num_leaves=50, subsample=0.6).fit(x_train, y_train),
'cat after_grid' : CatBoostClassifier(random_state=42, verbose=0, depth= 4, iterations= 200, l2_leaf_reg= 5, learning_rate=0.1).fit(x_train, y_train)}

print(model_f1(after_grid))
print(model_accuracy(after_grid))

/home/honor/Programing/my_env/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


xgb_after_grid: 0.737516005121639
lg_after_grid: 0.6498455200823893
dc after_grid: 0.7021028037383178
lgbm after_grid: 0.7196652719665272
cat after_grid: 0.7166077738515901
None
xgb_after_grid: 0.8906666666666667
lg_after_grid: 0.8186666666666667
dc after_grid: 0.864
lgbm after_grid: 0.8928
cat after_grid: 0.8930666666666667
None


In [14]:
grid_models = {'xgb_after_grid' : XGBClassifier(colsample_bytree = 0.8, learning_rate = 0.01, max_depth= 9, min_child_weight= 3, n_estimators= 100, scale_pos_weight=3, subsample= 0.8).fit(x_train, y_train),
'rf_after_grid' : RandomForestClassifier(class_weight= 'balanced_subsample', max_depth= 14, min_samples_leaf= 2, min_samples_split= 12, n_estimators= 150, random_state=42).fit(x_train, y_train),
'dc_after_grid' : DecisionTreeClassifier(class_weight='balanced', criterion='entropy', max_depth=7, min_samples_leaf=2,  min_samples_split=25).fit(x_train, y_train),
'lgbm_after_grid': LGBMClassifier(random_state=42, verbose=-1, learning_rate= 0.1, max_depth= 5, n_estimators= 100, num_leaves=50, subsample=0.6).fit(x_train, y_train),
'cat_after_grid' : CatBoostClassifier(random_state=42, verbose=0, depth= 4, iterations= 200, l2_leaf_reg= 5, learning_rate=0.1).fit(x_train, y_train)}
def Possible_Overfitting(x_train, x_test, y_train, y_test):
    for name, model in grid_models.items():
        train_preds = model.predict(x_train)
        print(f'{name} : {accuracy_score(y_train, train_preds)}')

    for name, model in grid_models.items():
        test_preds = model.predict(x_test)
        print(f'{name}, {accuracy_score(y_test, test_preds)}')

Possible_Overfitting(x_train, x_test, y_train, y_test)

xgb_after_grid : 0.9284444444444444
rf_after_grid : 0.9392
dc_after_grid : 0.8864888888888889
lgbm_after_grid : 0.9249777777777778
cat_after_grid : 0.9167111111111111
xgb_after_grid, 0.8904
rf_after_grid, 0.8864
dc_after_grid, 0.864
lgbm_after_grid, 0.8928
cat_after_grid, 0.8930666666666667


In [15]:
after_overfitting = {'xgb_after_overfitting' : XGBClassifier(colsample_bytree = 0.8, learning_rate = 0.01, max_depth= 9, min_child_weight= 3, n_estimators= 100, scale_pos_weight=3, subsample= 0.8).fit(x_train, y_train),
'rf_after_overfitting' : RandomForestClassifier(class_weight= 'balanced_subsample', max_depth= 14, min_samples_leaf= 2, min_samples_split= 12, n_estimators= 150, random_state=42).fit(x_train, y_train),
'dc after_overfitting' : DecisionTreeClassifier(class_weight='balanced', criterion='entropy', max_depth=7, min_samples_leaf=2,  min_samples_split=25).fit(x_train, y_train),
'lgbm after_grid': LGBMClassifier(random_state=42, verbose=-1, learning_rate= 0.1, max_depth= 5, n_estimators= 100, num_leaves=50, subsample=0.6).fit(x_train, y_train),
'cat after_grid' : CatBoostClassifier(random_state=42, verbose=0, depth= 4, iterations= 200, l2_leaf_reg= 5, learning_rate=0.1).fit(x_train, y_train)}

print(model_f1(after_overfitting))
print(model_accuracy(after_overfitting))

xgb_after_overfitting: 0.7387158296249206
rf_after_overfitting: 0.7373612823674476
dc after_overfitting: 0.7021028037383178
lgbm after_grid: 0.7196652719665272
cat after_grid: 0.7166077738515901
None
xgb_after_overfitting: 0.8904
rf_after_overfitting: 0.8864
dc after_overfitting: 0.864
lgbm after_grid: 0.8928
cat after_grid: 0.8930666666666667
None


In [40]:
xgb_after_overfitting = XGBClassifier(colsample_bytree = 0.8, learning_rate = 0.01, max_depth= 9, min_child_weight= 3, n_estimators= 100, scale_pos_weight=3, subsample= 0.8).fit(x_train, y_train)
rf_after_overfitting = RandomForestClassifier(class_weight= 'balanced_subsample', max_depth= 14, min_samples_leaf= 2, min_samples_split= 12, n_estimators= 150, random_state=42).fit(x_train, y_train)
dc_after_overfitting = DecisionTreeClassifier(class_weight='balanced', criterion='entropy', max_depth=7, min_samples_leaf=2,  min_samples_split=25).fit(x_train, y_train)
lgbm_after_overfitting = LGBMClassifier(random_state=42, verbose=-1, learning_rate= 0.1, max_depth= 5, n_estimators= 100, num_leaves=50, subsample=0.6).fit(x_train, y_train)
cat_after_overfitting = CatBoostClassifier(random_state=42, verbose=0, depth= 4, iterations= 200, l2_leaf_reg= 5, learning_rate=0.1).fit(x_train, y_train)


In [ ]:
y_proba = xgb_after_overfitting.predict_proba(x_test)[:, 1]
threshold = 0.5
xgb_pred_new = (y_proba >= threshold).astype(int)
print('xgb f1_score after treshhold:' , f1_score(y_test, xgb_pred_new))
print('xgb accuracy  after treshhold:' , accuracy_score(y_test, xgb_pred_new))

xgb f1_score after treshhold: 0.7387158296249206
xgb accuracy  after treshhold: 0.8904


In [65]:
y_proba = rf_after_overfitting.predict_proba(x_test)[:, 1]
threshold = 0.49
rf_pred_new = (y_proba >= threshold).astype(int)
print('rf f1_score after treshhold:' , f1_score(y_test, rf_pred_new))
print('rf accuracy  after treshhold:' , accuracy_score(y_test, rf_pred_new))

rf f1_score after treshhold: 0.7400611620795107
rf accuracy  after treshhold: 0.8866666666666667


In [53]:
y_proba = dc_after_overfitting.predict_proba(x_test)[:, 1]
threshold = 0.7
dc_pred_new = (y_proba >= threshold).astype(int)
print('lg f1_score after treshhold:' , f1_score(y_test, dc_pred_new))
print('lg accuracy  after treshhold:' , accuracy_score(y_test, dc_pred_new))

lg f1_score after treshhold: 0.7113133940182055
lg accuracy  after treshhold: 0.8816


In [53]:
y_proba = lgbm_after_overfitting.predict_proba(x_test)[:, 1]
threshold = 0.37
lgbm_pred_new = (y_proba >= threshold).astype(int)
print('xgb f1_score after treshhold:' , f1_score(y_test, lgbm_pred_new))
print('xgb accuracy  after treshhold:' , accuracy_score(y_test, lgbm_pred_new))

xgb f1_score after treshhold: 0.7323037323037324
xgb accuracy  after treshhold: 0.8890666666666667


In [34]:
y_proba = cat_after_overfitting.predict_proba(x_test)[:, 1]
threshold = 0.35
cat_pred_new = (y_proba >= threshold).astype(int)
print('xgb f1_score after treshhold:' , f1_score(y_test, cat_pred_new))
print('xgb accuracy  after treshhold:' , accuracy_score(y_test, cat_pred_new))

xgb f1_score after treshhold: 0.734977862112587
xgb accuracy  after treshhold: 0.8882666666666666


In [ ]:
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import f1_score, accuracy_score

# 1. CatBoost
cat_model = CatBoostClassifier(random_state=42, verbose=0)
cat_model.fit(x_train, y_train)
cat_pred = cat_model.predict(x_test)

print(f'CatBoost F1: {f1_score(y_test, cat_pred)}')
print(f'CatBoost Accuracy: {accuracy_score(y_test, cat_pred)}')

# 2. LightGBM
lgb_model = LGBMClassifier(random_state=42)
lgb_model.fit(x_train, y_train)
lgb_pred = lgb_model.predict(x_test)

print(f'LightGBM F1: {f1_score(y_test, lgb_pred)}')
print(f'LightGBM Accuracy: {accuracy_score(y_test, lgb_pred)}')


CatBoost F1: 0.7135819845179451
CatBoost Accuracy: 0.8914666666666666
[LightGBM] [Info] Number of positive: 2266, number of negative: 8984
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000313 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 847
[LightGBM] [Info] Number of data points in the train set: 11250, number of used features: 12
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.201422 -> initscore=-1.377429
[LightGBM] [Info] Start training from score -1.377429
LightGBM F1: 0.7196652719665272
LightGBM Accuracy: 0.8928


In [80]:
pip install lightgbm

  Using cached lightgbm-4.7.0-py3-none-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (18 kB)
Using cached lightgbm-4.7.0-py3-none-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (3.5 MB)
Note: you may need to restart the kernel to use updated packages.
